# Multiverse of Monsoon — clean Colab workflow

Run cells in order. The broad benchmark is a 2020–21 hold-out proxy, not the hidden organiser test.

In [ ]:
!nvidia-smi
!mkdir -p data weights
!wget -q https://github.com/OfficialAbhinavSingh/yug-monsoon-kit/releases/download/v1.0.0/monsoon_r1_kit-1.0.0-py3-none-any.whl -O data/monsoon_r1_kit-1.0.0-py3-none-any.whl
!wget -q https://github.com/OfficialAbhinavSingh/yug-monsoon-kit/releases/download/v1.0.0/india_climate_train.nc -O data/india_climate_train.nc
!wget -q https://github.com/OfficialAbhinavSingh/yug-monsoon-kit/releases/download/v1.0.0/stats.json -O data/stats.json
!pip install -q data/monsoon_r1_kit-1.0.0-py3-none-any.whl xarray netCDF4 scipy pandas

In [ ]:
# Upload colab_code_v5.zip from this repository. Do NOT delete weights/ here.
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -o -q "{zip_name}"
!grep -n "IN_CHANNELS = 42" model/architecture.py
!python -m unittest discover -s tests -v
!python -m train.prepare_data

In [ ]:
# Primary submission candidate.
!python -m train.train --arch unet --seed 42 --save-as-default
!cp weights/model.pt weights/model_seed42_candidate.pt
!python -m train.eval_local --ckpt weights/model.pt
!python -m train.stress_benchmark --ckpt weights/model.pt

In [ ]:
# Optional second seed. Keep it only if BOTH evaluations improve.
!python -m train.train --arch unet --seed 101 --out-name model_ens2.pt
!python -m train.eval_local --ckpt weights/model.pt --ens-ckpt weights/model_ens2.pt
!python -m train.stress_benchmark --ckpt weights/model.pt --ens-ckpt weights/model_ens2.pt

In [ ]:
# API contract check through the actual server entrypoint.
!python serve.py --port 8000 > /tmp/monsoon_server.log 2>&1 &
!sleep 5
!python -m monsoon_r1.schema_validator http://127.0.0.1:8000
!cat /tmp/monsoon_server.log

In [ ]:
# Download primary candidate and required inference artifacts.
from google.colab import files
for path in ['weights/model.pt', 'weights/model_seed42_candidate.pt', 'weights/model_ens2.pt', 'weights/climatology.npy', 'weights/static.npy', 'weights/model_meta.json']:
    try:
        files.download(path)
    except FileNotFoundError:
        print('Optional artifact absent:', path)